# Autoencoders
**Machine Learning · Week 6 (Learning without labels) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/ml/w6d4.html

An **autoencoder** is a neural network trained to copy its input through a narrow middle, the
**code**. To copy well it must squeeze out what matters, so it learns its own compression, without labels.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.set_num_threads(1)
digits = load_digits()
X = digits.data / 16.0                                # 1797 images, 64 pixel values in [0, 1]
X_train, X_test, y_train, y_test = train_test_split(X, digits.target, test_size=0.25, random_state=0, stratify=digits.target)
T = lambda a: torch.tensor(a, dtype=torch.float32)

def train_autoencoder(inputs, targets, code_size=2, epochs=1500, seed=1):
    """Encoder 64 -> 32 -> code_size, decoder code_size -> 32 -> 64. Learns to turn inputs into targets."""
    torch.manual_seed(seed)
    encoder = torch.nn.Sequential(torch.nn.Linear(64, 32), torch.nn.Tanh(), torch.nn.Linear(32, code_size))
    decoder = torch.nn.Sequential(torch.nn.Linear(code_size, 32), torch.nn.Tanh(), torch.nn.Linear(32, 64), torch.nn.Sigmoid())
    optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=3e-3)
    a, b = T(inputs), T(targets)
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = torch.nn.functional.mse_loss(decoder(encoder(a)), b)   # how well does it copy?
        loss.backward()
        optimizer.step()
    return encoder, decoder

def errors(encoder, decoder, A):
    """Reconstruction error of every image: the mean squared difference over its 64 pixels."""
    with torch.no_grad():
        return ((decoder(encoder(T(A))).numpy() - A) ** 2).mean(axis=1)

encoder, decoder = train_autoencoder(X_train, X_train)          # input = target: no labels anywhere
print(f"reconstruction error: training digits {errors(encoder, decoder, X_train).mean():.4f}, "
      f"new test digits {errors(encoder, decoder, X_test).mean():.4f}")

## The learned 2-number code
Nobody told the network which digit is which. The colors use the labels only to show the result.

In [ ]:
with torch.no_grad():
    codes = encoder(T(X_train)).numpy()
fig, ax = plt.subplots(figsize=(5.4, 4.2), layout="constrained")
points = ax.scatter(codes[:, 0], codes[:, 1], c=y_train, cmap="tab10", s=7)
fig.colorbar(points, ax=ax, label="digit", ticks=range(10))
ax.set_title("Digits in the autoencoder's 2D code")
plt.show()

## Against PCA
With the same number of numbers kept, the autoencoder's bends let it rebuild new digits better than
PCA's straight-line compression. (With only linear layers, it would find PCA's answer.)

In [ ]:
from sklearn.decomposition import PCA

for k in [2, 4, 8]:
    enc_k, dec_k = (encoder, decoder) if k == 2 else train_autoencoder(X_train, X_train, code_size=k)
    pca = PCA(n_components=k).fit(X_train)
    pca_error = ((pca.inverse_transform(pca.transform(X_test)) - X_test) ** 2).mean()
    print(f"{k} numbers per digit: PCA test error {pca_error:.4f}, autoencoder test error {errors(enc_k, dec_k, X_test).mean():.4f}")

pca2 = PCA(n_components=2).fit(X_train)
with torch.no_grad():
    ae_back = decoder(encoder(T(X_test[:8]))).numpy()
pca_back = pca2.inverse_transform(pca2.transform(X_test[:8]))
fig, axes = plt.subplots(3, 8, figsize=(7.5, 3), layout="constrained")
for col in range(8):
    for row, image in enumerate([X_test[col], pca_back[col], ae_back[col]]):
        axes[row, col].imshow(image.reshape(8, 8), cmap="Purples", vmin=0, vmax=1)
        axes[row, col].axis("off")
axes[0, 0].set_title("original", loc="left", fontsize=10)
axes[1, 0].set_title("PCA, 2 numbers", loc="left", fontsize=10)
axes[2, 0].set_title("autoencoder, 2 numbers", loc="left", fontsize=10)
plt.show()

## Spotting the unusual
Trained on normal digits, the autoencoder rebuilds normal digits well and strange inputs badly.
Here: new test digits, the same digits mirrored left to right, and random noise. Set the alarm so that
5% of normal digits raise it, and count what gets caught. (Use new normal data for this, not the
training digits: those are rebuilt better than anything new.)

In [ ]:
mirrored = X_test.reshape(-1, 8, 8)[:, :, ::-1].reshape(-1, 64)
rng = np.random.default_rng(0)
noise = rng.random((200, 64)) * (rng.random((200, 64)) < 0.45)
e_test, e_mirror, e_noise = errors(encoder, decoder, X_test), errors(encoder, decoder, mirrored), errors(encoder, decoder, noise)

threshold = np.quantile(e_test, 0.95)
bins = np.linspace(0, 0.25, 50)
plt.hist(e_test, bins=bins, alpha=0.7, color="#b9a7ff", label="normal test digits")
plt.hist(e_mirror, bins=bins, alpha=0.7, color="#ff8a50", label="mirrored digits")
plt.hist(e_noise, bins=bins, alpha=0.7, color="#8a85a3", label="noise")
plt.axvline(threshold, color="#e5566b", lw=2, label="alarm threshold")
plt.xlabel("reconstruction error")
plt.ylabel("number of images")
plt.legend()
plt.show()
print(f"caught: {np.mean(e_mirror > threshold):.0%} of mirrored digits, {np.mean(e_noise > threshold):.0%} of noise images "
      f"(with 5% false alarms)")
print("mirrored digits caught, by digit:", "  ".join(f"{d}: {np.mean(e_mirror[y_test == d] > threshold):.0%}" for d in range(10)))

Mirrored 0s and 8s almost never raise the alarm: mirrored, they still look like normal 0s and 8s.
Mirrored 6s are caught most often. The alarm only knows "unlike what I learned", not "wrong".

## Your turn: a denoising autoencoder
Train the autoencoder to map **noisy** digits to the **clean** ones. Add Gaussian noise (standard
deviation 0.25, then clip to [0, 1]) to the training digits, train with `code_size=8`, and clean up
noisy test digits.

In [ ]:
rng = np.random.default_rng(1)
noisy_train = X_train          # TODO: X_train + 0.25 * rng.standard_normal(X_train.shape), clipped to [0, 1]
noisy_test = X_test            # TODO: the same for X_test
enc_d, dec_d = None, None      # TODO: train_autoencoder(noisy_train, X_train, code_size=8)

if dec_d is not None:
    with torch.no_grad():
        cleaned = dec_d(enc_d(T(noisy_test[:8]))).numpy()
    fig, axes = plt.subplots(3, 8, figsize=(7.5, 3), layout="constrained")
    for col in range(8):
        for row, image in enumerate([X_test[col], noisy_test[col], cleaned[col]]):
            axes[row, col].imshow(image.reshape(8, 8), cmap="Purples", vmin=0, vmax=1)
            axes[row, col].axis("off")
    plt.show()

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
rng = np.random.default_rng(1)
noisy_train = np.clip(X_train + 0.25 * rng.standard_normal(X_train.shape), 0, 1)
noisy_test = np.clip(X_test + 0.25 * rng.standard_normal(X_test.shape), 0, 1)
enc_d, dec_d = train_autoencoder(noisy_train, X_train, code_size=8)     # noisy in, clean out

with torch.no_grad():
    cleaned = dec_d(enc_d(T(noisy_test[:8]))).numpy()
fig, axes = plt.subplots(3, 8, figsize=(7.5, 3), layout="constrained")
for col in range(8):
    for row, image in enumerate([X_test[col], noisy_test[col], cleaned[col]]):
        axes[row, col].imshow(image.reshape(8, 8), cmap="Purples", vmin=0, vmax=1)
        axes[row, col].axis("off")
axes[0, 0].set_title("clean", loc="left", fontsize=10)
axes[1, 0].set_title("noisy input", loc="left", fontsize=10)
axes[2, 0].set_title("denoised", loc="left", fontsize=10)
plt.show()
with torch.no_grad():
    after = ((dec_d(enc_d(T(noisy_test))).numpy() - X_test) ** 2).mean()
print(f"error against the clean digits: noisy input {((noisy_test - X_test) ** 2).mean():.4f}, denoised {after:.4f}")